In [1]:
print("hello")

hello


Practical Task: End-to-End ML Model Deployment and Tracking

Build a simple Iris classification pipeline integrating FastAPI, MLflow, DVC, and AWS SageMaker.

Requirements:

1. ML Model: Train a Random Forest classifier using the Iris dataset and save the trained model as a `.pkl` file.
2. MLflow: Track model parameters, accuracy, and experiment metrics using MLflow.
3. DVC: Initialize DVC and track the dataset and model training pipeline.
4. FastAPI: Create a `/predict` POST endpoint that accepts flower features and returns the predicted species.
5. AWS SageMaker: Prepare a SageMaker deployment configuration for hosting the trained model.

Expected output: A working FastAPI prediction endpoint, MLflow experiment logs, DVC-tracked pipeline, and SageMaker deployment configuration.

Constraint: Focus on implementation and integration. Use an existing dataset; avoid spending time on extensive preprocessing or hyperparameter tuning.

```
git init
dvc init

In [ ]:
import joblib
import pandas as pd
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("iris.csv")
X, y = df.drop(columns=["label"]), df["label"]
X_train, y_train, X_test, y_test = train_test_split(X, y, test_size=0.2)

para = {"n_estimators": 100}

mlflow.set_experiment()

with mlflow.start_run():
    model = RandomForestClassifier(**params)

    model.fit(X_train)
    acc = model.score(model.predict(X_test), y_test) 

    mlflow.log_params(para)
    mlflow.metrics("accuracy", acc)
    mlflow.skleanrn.log_model(model)

    joblib.dump(model, "model.joblib")

```yaml
start:
    runs: python train.py
        devs: 
         - train.py
         - model.joblib
        outs:
         - 

```
dvc repro

git add .
git commit -m "train model"
git remote add origin "
git branch -M main
git push -u origin main

In [ ]:
from fastapi import FastAPI, jsonify, requests
import joblib

app = FastAPI()

@app.post("/predict")
def predict():
    data = requests()[0]
    model = joblib.load("model.joblib")
    res = model.predict(data)
    return jsonify({"response": res})

app.run_server("5000:5000")

In [ ]:
import json
import os
import joblib
import numpy as np


def model_fn(model_dir):
    """
    Load the trained model from the model_dir directory.
    SageMaker unpacks the model.tar.gz file into model_dir.
    """
    model_path = os.path.join(model_dir, "model.joblib")
    if not os.path.exists(model_path):
        # Fallback if model was saved directly inside model_dir
        model_path = os.path.join(model_dir, "iris_model.joblib")
    
    model = joblib.load(model_path)
    return model


def input_fn(request_body, request_content_type):
    """
    Deserialize and prepare the incoming request data.
    Supports application/json format.
    """
    if request_content_type == "application/json":
        data = json.loads(request_body)
        # Expects {"inputs": [[5.1, 3.5, 1.4, 0.2], ...]} or similar list structure
        if "inputs" in data:
            features = np.array(data["inputs"])
        else:
            features = np.array(data)
        return features
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")


def predict_fn(input_data, model):
    """
    Run prediction using the loaded model and preprocessed input data.
    """
    prediction = model.predict(input_data)
    # If the model supports probabilities, you can also return model.predict_proba(input_data)
    return prediction


def output_fn(prediction, accept):
    """
    Format the prediction result to return to the client.
    Supports application/json.
    """
    if accept == "application/json" or accept == "*/*":
        output = {"predictions": prediction.tolist()}
        return json.dumps(output), "application/json"
    else:
        raise ValueError(f"Unsupported accept type: {accept}")



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder.appName("EcommerceAnalysis").getOrCreate()

data = [
    (101, 1, "Electronics", 2, 15000, "2026-01-05", "Chennai"),
    (102, 2, "Clothing", 3, 1200, "2026-01-06", "Bangalore"),
    (103, 3, "Electronics", 1, 25000, "2026-01-07", "Thrissur"),
    (104, 4, "Furniture", 2, 8000, "2026-01-08", "Kochi"),
    (105, 1, "Clothing", 5, 900, "2026-01-09", "Chennai"),
    (106, 5, "Electronics", 1, 45000, "2026-01-10", "Bangalore"),
    (107, 2, "Furniture", 1, 12000, "2026-01-11", "Kochi"),
    (108, 6, "Clothing", 4, 1500, "2026-01-12", "Thrissur"),
    (109, 3, "Electronics", 2, 18000, "2026-01-13", "Chennai"),
    (110, 7, "Furniture", 3, 6000, "2026-01-14", "Bangalore"),
    (111, 4, "Electronics", 1, 30000, "2026-01-15", "Kochi"),
    (112, 5, "Clothing", 6, 800, "2026-01-16", "Thrissur"),
    (113, 8, "Furniture", 2, 9500, "2026-01-17", "Chennai"),
    (114, 6, "Electronics", 1, 22000, "2026-01-18", "Bangalore"),
    (115, 7, "Clothing", 4, 1100, "2026-01-19", "Kochi"),
    (116, 9, "Electronics", 2, 16000, "2026-01-20", "Thrissur"),
    (117, 8, "Furniture", 1, 14000, "2026-01-21", "Chennai"),
    (118, 10, "Clothing", 5, 1300, "2026-01-22", "Bangalore"),
    (119, 9, "Electronics", 1, 35000, "2026-01-23", "Kochi"),
    (120, 10, "Furniture", 2, 7500, "2026-01-24", "Thrissur"),
]

columns = [
    "Order_ID",
    "Customer_ID",
    "Product_Category",
    "Quantity",
    "Unit_Price",
    "Order_Date",
    "City"
]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/31 16:35:32 WARN Utils: Your hostname, Orbins-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.40 instead (on interface en0)
26/10/31 16:35:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/31 16:35:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------+-----------+----------------+--------+----------+----------+---------+
|Order_ID|Customer_ID|Product_Category|Quantity|Unit_Price|Order_Date|     City|
+--------+-----------+----------------+--------+----------+----------+---------+
|     101|          1|     Electronics|       2|     15000|2026-01-05|  Chennai|
|     102|          2|        Clothing|       3|      1200|2026-01-06|Bangalore|
|     103|          3|     Electronics|       1|     25000|2026-01-07| Thrissur|
|     104|          4|       Furniture|       2|      8000|2026-01-08|    Kochi|
|     105|          1|        Clothing|       5|       900|2026-01-09|  Chennai|
|     106|          5|     Electronics|       1|     45000|2026-01-10|Bangalore|
|     107|          2|       Furniture|       1|     12000|2026-01-11|    Kochi|
|     108|          6|        Clothing|       4|      1500|2026-01-12| Thrissur|
|     109|          3|     Electronics|       2|     18000|2026-01-13|  Chennai|
|     110|          7|      

Practical Task: E-Commerce Sales Analysis Using Apache Spark (PySpark)

Dataset: Create a PySpark DataFrame with 20 records and the following columns:

- Order_ID
- Customer_ID
- Product_Category
- Quantity
- Unit_Price
- Order_Date
- City

Requirements:

1. DataFrame Operations: Load the dataset into a PySpark DataFrame and display its schema.
2. Data Cleaning: Identify and remove duplicate records and handle missing values.
3. Transformations: Create a new column `Total_Amount` using Quantity × Unit_Price.
4. Aggregations: Calculate total revenue and total orders for each Product_Category.
5. Filtering: Identify customers whose total purchase amount exceeds ₹10,000.
6. Sorting: Display the top 5 customers based on total purchase amount.

Expected output: A PySpark script demonstrating DataFrame operations, transformations, filtering, aggregations, and sorting.

In [ ]:
from pyspark.sql import functions as F

df = df.drop_duplicates()

df = df.withColumn("Total_Amount", df['Quantity'] * df['Unit_Price'])

results = df.groupBy("Product_Category").agg(
    F.sum(F.col("Total_Amount").alias("Total_Revenue")),
    F.count(F.col("*")).alias("Total_Orders")
)

results.show()

filtered = df.filter(F.col("Total_Amount") > 10000)

filtered.show()

sorted = df.sort(F.desc(df["Total_Amount"])).show(5)

+----------------+----------------------------------+------------+
|Product_Category|sum(Total_Amount AS Total_Revenue)|Total_Orders|
+----------------+----------------------------------+------------+
|     Electronics|                            255000|           8|
|        Clothing|                             29800|           6|
|       Furniture|                             94000|           6|
+----------------+----------------------------------+------------+

+--------+-----------+----------------+--------+----------+----------+---------+------------+
|Order_ID|Customer_ID|Product_Category|Quantity|Unit_Price|Order_Date|     City|Total_Amount|
+--------+-----------+----------------+--------+----------+----------+---------+------------+
|     101|          1|     Electronics|       2|     15000|2026-01-05|  Chennai|       30000|
|     103|          3|     Electronics|       1|     25000|2026-01-07| Thrissur|       25000|
|     104|          4|       Furniture|       2|      8000|2

26/10/31 19:49:17 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1036869 ms exceeds timeout 120000 ms
26/10/31 19:49:17 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/31 19:49:45 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stor